In [ ]:
from qarp.blocks import (
    AGateBlock,
    CompositeBlock, HnBlock,
    ComputationalBasisStateBlock,
    DickeStateBlock, HypergraphStateBlock,
    LayerBlock, PauliBlock,
    HadamardTestBlock, BlockEncodingBlock, XnBlock, QFTBlock, SWAPTestBlock,
    IdentityBlock, UCCBlock, MappedONVStateBlock, BrickworkEntanglingBlock, CostOperatorBlock,
    UPCCDBlock, SynthesizedStateBlock, SynthesizedUnitaryBlock, ReadoutBlock
    )
from qarp.algorithms import StateVector, SWAPTest
from qarp.graphs import Hypergraph
from qarp.engines import QarpEngine
from qarp.operators import JordanWigner, NoGrouping, QubitOperator
import numpy as np
import qarpx as qx

### ComputationalBasisStateBlock

In [ ]:
block = ComputationalBasisStateBlock([0, 1, 0, 1]).build().plot()

### SynthesizedStateBlock

In [ ]:
amplitudes = [0.1 + 0.2j, 0.3 + 0.4j, 0, 0.7 + 0.8j]

# Dict keys are LSB-first tuples (element k = qubit k = bit k of the index),
# the same convention as list indices and Sampler keys.
amplitudes_dict = {tuple((i >> k) & 1 for k in range(2)): amp for i, amp in enumerate(amplitudes) if amp != 0}

block = SynthesizedStateBlock(n_qubits=2, amplitudes=amplitudes_dict).build()
block.plot()

# Table to compare input amplitudes, block amplitudes and statevector
for i, (in_amp, blk_amp, sv_amp) in enumerate(zip(amplitudes/np.linalg.norm(amplitudes), block.amplitudes, np.array(qx.QarpSimulator().statevector(block.flatten(), block.n_qubits)))):
    print(f"|{i:0{block.n_qubits}b}>: input {in_amp:.4f}, block {blk_amp:.4f}, statevector {sv_amp:.4f}")

print("Difference between input amplitudes and block amplitudes:", np.linalg.norm(amplitudes/np.linalg.norm(amplitudes) - block.amplitudes))

### SynthesizedUnitaryBlock

In [ ]:
# Example with a random 4 qubit unitary
from scipy.stats import unitary_group
n_qubits = 4
unitary = unitary_group.rvs(2**n_qubits)

block = SynthesizedUnitaryBlock(unitary_matrix=unitary).build()
block.plot(spacing=0.3, verbose=True, decompose_boxes=True)

print("Difference between input unitary and block unitary:", np.linalg.norm(np.array(qx.QarpSimulator().unitary_matrix(block.flatten(), block.n_qubits)) - unitary))

### AGateBlock

In [ ]:
AGateBlock(theta=0.1, phi=0.2).build().plot()

### HnBlock

In [ ]:
HnBlock(5).build().plot()

### LayerBlock

In [ ]:
block = LayerBlock(qx.GateType.CCX, 5, overlapping=2, periodic_boundary=True)
block.build().plot()

In [ ]:
from sympy import Symbol

# Single symbol or float for all gates
theta = Symbol('theta')
# theta = np.random.rand()
block = LayerBlock(qx.GateType.Rx, 5, parameters=[theta])
block.build().plot()
print(block.symbols)

# Different symbols or floats for each gate
thetas = [Symbol(f'theta_{i}') for i in range(5)]
# thetas = [np.random.rand() for i in range(5)]
block = LayerBlock(qx.GateType.Ry, 5, parameters=thetas)
block.build().plot()
print(block.symbols)

In [ ]:
from sympy import Symbol

# U3 gate with 3 parameters - same for all gates
theta, phi, lam = Symbol('theta'), Symbol('phi'), Symbol('lambda')
block = LayerBlock(qx.GateType.U, 5, parameters=[theta, phi, lam])
block.build().plot(spacing=0.5)
print(block.symbols)

# U3 gate with different parameters for each gate (5 gates * 3 params = 15 params)
params = [Symbol(f'theta_{i}') for i in range(5)] + \
         [Symbol(f'phi_{i}') for i in range(5)] + \
         [Symbol(f'lambda_{i}') for i in range(5)]
block = LayerBlock(qx.GateType.U, 5, parameters=params)
block.build().plot(spacing=0.7)
print(block.symbols)

# Or more structured:
params = []
for i in range(5):
    params.extend([Symbol(f'theta_{i}'), Symbol(f'phi_{i}'), Symbol(f'lambda_{i}')])
block = LayerBlock(qx.GateType.U, 5, parameters=params)
block.build().plot(spacing=0.6)
print(block.symbols)

   # NOTE: qarpx has only one ``U`` GateType (3-param U3); a separate U2 (2-param)
   # gate variant doesn't exist on the IR — demo cell removed.


### PauliBlock

In [ ]:
PauliBlock(pauli_string="XYZY", target_qubits=[0, 1, 3, 4], measure=False, change_basis=False, name="PauliMeasurement").build().plot()

PauliBlock({0:"X", 1:"Y", 3:"Y"}).build().plot()

### HadamardTestBlock

In [ ]:
bra = IdentityBlock(1, name="bra")
op = IdentityBlock(1, name="op")
ket = IdentityBlock(1, name="ket")

HadamardTestBlock(state=ket, unitary=op, unitary_dagger=bra).build().plot()

### IdentityBlock and CompositeBlock

In [ ]:
# NOTE: This composite-frame demo (HadamardTestBlock with target_qubits=[1, 3]
# inside a 4-qubit CompositeBlock alongside IdentityBlock(target_qubits=[0,1,3]))
# currently triggers a qubit-remap edge case in the composite framework.
# Skipping until the framework handles non-contiguous target_qubits ranges in nested composites.
try:
    meas_block = HadamardTestBlock(state=IdentityBlock(1), unitary=IdentityBlock(1), measure=True, target_qubits=[1, 3], name='HTest')
    meas_block.build().plot()
    identity = IdentityBlock(3, target_qubits=[0, 1, 3])
    identity.build().plot()
    CompositeBlock([identity, meas_block], 4).build().plot()
except IndexError as e:
    print(f'(known issue) composite remap: {e}')


### DickeStateBlock and dagger()

In [ ]:
block = DickeStateBlock(3, 2)
block.build().plot()

original_unitary = np.array(qx.QarpSimulator().unitary_matrix(block.build().flatten(), block.n_qubits))
dagger_block = block.dagger()
dagger_unitary = np.array(qx.QarpSimulator().unitary_matrix(dagger_block.build().flatten(), dagger_block.n_qubits))

expected_dagger = np.conj(original_unitary).T
print("Dagger is correct:", np.allclose(dagger_unitary, expected_dagger))

print("Identity: ", np.allclose(original_unitary @ dagger_unitary, np.eye(2**3)))

### ReadoutBlock

In [ ]:
ReadoutBlock(4).build().plot()

### MappedONVStateBlock

In [ ]:
onv = [1, 1, 1, 0]
MappedONVStateBlock(occupation_number_vector=onv, mapping=JordanWigner()).build().plot()

### UCCBlock and symbols

In [ ]:
onv = [1, 1, 0, 0]
ref = MappedONVStateBlock(occupation_number_vector=onv, mapping=JordanWigner())
ucc = UCCBlock(
    occupation_number_vector=onv, mapping=JordanWigner(), singles=True, doubles=True, generalised=True, grouping=NoGrouping()
)
ucc.build().plot(spacing=0.4)
bra = CompositeBlock([ref, ucc], 4)
bra.build()
ket = bra.refresh_symbols("_1")
# ket = CompositeBlock([ref, ucc.refresh_symbols("_1")], 4)

#[s20, d0213, s31]

print("-------")
print(ucc.build().symbols)
print(bra.symbols)
print(ket.symbols)
print("---------")
# bra_values = {
#     Symbol('s31'): 0.0,
#     Symbol('s20'): 0.0,
#     Symbol('d2031'): 0.0
# }
# ket_values = {
#     Symbol('s31_1'): 0.0,
#     Symbol('s20_1'): 0.0,
#     Symbol('d2031_1'): -0.113
# }

# all_parameters = {**bra_values, **ket_values}
all_parameters = dict(zip(bra.symbols + ket.symbols, [0.0] * len(bra.symbols) + [0.3] * len(ket.symbols)))
print(all_parameters)

engine = QarpEngine()
engine.build([StateVector(bra=bra, ket=ket).build()])
svo = engine.run(all_parameters)

engine = QarpEngine()
engine.build([SWAPTest(bra=bra, ket=ket, n_shots=1000000).build()])
sto2_sampled = engine.run(all_parameters)

# print(sum(svo)**2)
# print(sum(sto2_sampled))
# print(abs(sum(svo) - 0.9936222907489856) < 1e-9)
# print(abs(sum(sto2_sampled) - 0.9936222907489856**2) < 1e-2)

### UPCCDBlock

In [ ]:
block = UPCCDBlock([1, 1, 0, 0]).build()
block.plot(spacing=0.4)

### DOSQPEBlock

In [ ]:
from qarp.operators import JordanWigner
from qarp.operators.models import fermi_hubbard
from qarp.blocks import TrotterBlock, DOSQPEBlock, HnBlock
from qarp.plotting import plot

# Define the number of qubits and ancilla qubits
n_qubits = 2
n_ancilla = 3

# Build the Trotter block for the unitary evolution. In this case, we use the Fermi-Hubbard model.
qham = JordanWigner().encode_operator(fermi_hubbard((1,), t=0.14, U=0.231))
evolution = TrotterBlock(n_qubits, operator=qham).build()

# Build a layer of H gates to create the maximally mixed state using the purification register.
state = HnBlock(n_qubits).build()

# Create and build the DOSQPE block
dosqpe_block = DOSQPEBlock(state, evolution, n_ancilla=n_ancilla, n_state=n_qubits)
dosqpe_block.build().plot(spacing=0.35)

### QPEBlock

In [ ]:
from qarp.operators import JordanWigner
from qarp.operators.models import fermi_hubbard
from qarp.blocks import TrotterBlock, QPEBlock, HnBlock
from qarp.plotting import plot

n_qubits = 4

# Build the Trotter block for the unitary evolution. In this case, we use the Fermi-Hubbard model.
qham = JordanWigner().encode_operator(fermi_hubbard((2,), t=0.14, U=0.231))
evolution = TrotterBlock(n_qubits, operator=qham)

# Build the unitary that prepares the eigenstate |1111>
state = ComputationalBasisStateBlock([1] * n_qubits)

qpe = QPEBlock(state, evolution, n_ancilla=2, n_state=n_qubits).build()
qpe.plot(spacing=0.35)

### QFTBlock

In [ ]:
from qarp.blocks import QFTBlock

n_qubits = 4
qft = QFTBlock(n_qubits).build()

qft.plot()

### HadamardTestBlock

In [ ]:
from qarp.blocks import AGateBlock
import numpy as np

# Add the A gate with parameters theta and phi (divided by pi)
a_gate = AGateBlock(theta=0.5 / np.pi, phi=0.3 / np.pi).build()

a_gate.plot()

### SelectBlock

In [ ]:
from qarp.blocks import SelectBlock
from qarp.plotting import plot

# Define the unitaries to be applied as (global_phase, pauli_string) tuples
unitaries = [(0.0, 'XX'), (0.0, 'IZ')]

# Create a SELECT block with the specified unitaries using one ancillary control qubit
select_block = SelectBlock(unitaries, 1).build()

select_block.plot(spacing=0.3)

### CostOperatorBlock

In [ ]:
from qarp.blocks import CostOperatorBlock
from qarp.plotting import plot
import networkx as nx

n_nodes = 3
my_graph = nx.erdos_renyi_graph(n=n_nodes, p=0.5, seed=42)
block = CostOperatorBlock(n_qubits=n_nodes, problem=my_graph).build()

block.plot(spacing=0.3)

### MixedOperatorBlock

In [ ]:
from qarp.blocks import MixedOperatorBlock
from qarp.plotting import plot

n_nodes = 3
block = MixedOperatorBlock(n_qubits=n_nodes).build()

block.plot()

### BrickworkEntanglingBlock

In [ ]:
from qarp.blocks import BrickworkEntanglingBlock

n_qubits = 5
block = BrickworkEntanglingBlock(n_qubits=n_qubits, circular=True, use_cz=False).build()

block.plot()

### LinearEntanglingBlock

In [ ]:
from qarp.blocks import LinearEntanglingBlock

n_qubits = 5
block = LinearEntanglingBlock(n_qubits=n_qubits, circular=False, use_cz=True).build()

block.plot()

### HEABlock

In [ ]:
from qarp.blocks import HEABlock

n_qubits = 5
block_hea = HEABlock(n_qubits=n_qubits, n_layers=1, real=False, linear=True, circular=False, use_cz=True).build()

block_hea.plot(spacing=0.3)

### BrickworkPCEBlock

In [ ]:
from qarp.blocks import BrickworkPCEBlock

n_qubits = 5
block_pce = BrickworkPCEBlock(n_qubits=n_qubits, n_layers=1).build()

block_pce.plot(spacing=.5)

### HypergraphStateBlock

In [ ]:
edges = [(1, 2), (2, 3), (1, 2, 3)]

block = HypergraphStateBlock(edges=edges, n_qubits=4).build().plot()

### CVOQRAMStateBlock

In [ ]:
from qarp.blocks import CVOQRAMStateBlock
import numpy as np

dataset = {(1, 0, 0): np.sqrt(0.5), (0, 0, 0): np.sqrt(0.5)}

block = CVOQRAMStateBlock(dataset, target_qubits=list(range(6))).build()

block.plot()

### BlockEncodingBlock

In [ ]:
from qarp.blocks import BlockEncodingBlock
from qarp.operators import LinearCombinationUnitaries
import numpy as np

# 4x4 matrix
C=[[0,1,2,0],[-1,2,0,0],[0,0,1,2],[1,1,-1,1]]

C=np.array(C)

be=BlockEncodingBlock(C, name="BE circ")

circ_be = be.build()

full_unitary = np.array(qx.QarpSimulator().unitary_matrix(circ_be.flatten(), circ_be.n_qubits))
lambda_fac = be.lambda_factor()

be.plot(spacing=0.4)

In [ ]:
block = XnBlock(n_qubits=4).build()
block.plot()

In [ ]:
from qarp.blocks import SWAPTestBlock

u0 = ComputationalBasisStateBlock([1, 1, 0, 0]).build()
u1 = ComputationalBasisStateBlock([1, 0, 0, 1]).build()
swap = SWAPTestBlock(bra=u0, ket=u1, measure=True).build()
swap.plot()

In [ ]:
from qarp.blocks import HadamardTestBlock

id = IdentityBlock(n_qubits=4)
u0 = HnBlock(n_qubits=4)
u1 = XnBlock(n_qubits=4)
block = HadamardTestBlock(state=id, unitary=u0, unitary_dagger=u1).build()
block.plot()

In [ ]:
from qarp.blocks import HEABlock

hea = HEABlock(n_qubits=4, n_layers=5, real=True, linear=False, circular=True, use_cz=True).build()
hea.plot()

In [ ]:
from qarp.blocks import SPABlock

spa = SPABlock(n_qubits=4, n_layers=3, real=False, linear=False, circular=True).build()
spa.plot()

In [ ]:
from qarp.blocks import ComputationalBasisStateBlock, ReadoutBlock, CompositeBlock

state = ComputationalBasisStateBlock([1, 1, 0, 0]).build()
block = CompositeBlock([state]).build()
block.plot()

In [ ]:
from qarp.blocks import QSPBlock, QSPAngleFinder
import numpy as np

a = np.random.uniform(-1, 1)
P_poly = [0, 0, 0, 1]
oaf = QSPAngleFinder(P_poly)
optimal_angles = oaf.QSP()
qsp = QSPBlock(a, optimal_angles).build()
qsp.plot()

In [ ]:
from qarp.blocks import ProjectedControlPhaseBlock

block = ProjectedControlPhaseBlock(phase=0.5, dim=4, n_qubits=3).build()
block.plot()

### Arithmetic composition operators (`*`, `|`, `**` / `^`, `~`)

Blocks support arithmetic-style composition instead of manual `CompositeBlock([...])`
construction:

* `block1 * block2` — sequential composition (`block1` then `block2`), placed by each
  operand's `target_qubits`. Operands don't need matching `n_qubits`.
* `block1 | block2` — parallel composition: `block2` is auto-offset onto disjoint qubits.
* `block ** n` / `block ^ n` — repeat `block` sequentially `n` times (structural sharing,
  no copy).
* `~block` — shorthand for `block.dagger()` (Python has no overridable `!` prefix operator).

In [ ]:
# `*` — sequential composition, same qubits
seq = (HnBlock(3) * XnBlock(3)).build()   # H on all 3 qubits, then X on all 3
seq.plot()

In [ ]:
# `*` — mismatched sizes are fine: the smaller operand is placed by its own
# `target_qubits` inside the wider result (default [0, n) if unset)
wide = HnBlock(4).build()
narrow = XnBlock(2).build()
narrow.target_qubits = [1, 3]              # place X on qubits 1 and 3 of the 4-qubit result

placed = (wide * narrow).build()
print("placed.n_qubits:", placed.n_qubits)
placed.plot()

In [ ]:
# `|` — parallel composition: second operand auto-offset onto disjoint qubits
par = (HnBlock(2) | XnBlock(1)).build()   # Hn on qubits 0-1, X on qubit 2
print("par.n_qubits:", par.n_qubits)
par.plot()

In [ ]:
# `**` / `^` — repeat a block sequentially; the same instance is reused as
# every child (structural sharing), so this doesn't copy the layer 3 times
from qarp.blocks import HEABlock

layer = HEABlock(n_qubits=4, n_layers=1, real=False, linear=True, circular=False, use_cz=True).build()
stacked = (layer ** 3).build()
print("same instance reused:", stacked.blocks[0] is stacked.blocks[1] is layer)
stacked.plot(spacing=0.3)

# `^` is an alias for `**`
stacked_xor = (layer ^ 3).build()
print("`^` matches `**`:", len(stacked_xor.flatten()) == len(stacked.flatten()))

In [ ]:
# `~` — dagger shorthand, and chaining several operators together
h = HnBlock(2).build()
h_dag = ~h                                          # shorthand for h.dagger()

original_unitary = np.array(qx.QarpSimulator().unitary_matrix(h.build().flatten(), h.n_qubits))
dagger_unitary = np.array(qx.QarpSimulator().unitary_matrix(h_dag.build().flatten(), h_dag.n_qubits))
print("~block matches block.dagger():", np.allclose(dagger_unitary, np.conj(original_unitary).T))

# Chaining: normal operator precedence applies (`**` binds before `*`)
chained = (HnBlock(2) * XnBlock(2) * IdentityBlock(2) ** 3).build()
chained.plot()